In [ ]:
from serpapi import GoogleSearch

API_KEY = "YOUR_SERPAPI_KEY"

params = {
    "engine": "google",
    "q": 'site:linkedin.com/in "HR" "Jio"',
    "api_key": API_KEY,
    "num": 10
}

search = GoogleSearch(params)
results = search.get_dict()

for r in results.get("organic_results", []):
    print(r.get("link"))

https://in.linkedin.com/in/harjeetkhanduja
https://in.linkedin.com/in/mustafa-m-merchant
https://in.linkedin.com/in/manishsinghhr
https://in.linkedin.com/in/hr1505
https://in.linkedin.com/in/rahulmu
https://in.linkedin.com/in/sanjeev-varshney-0114b210
https://in.linkedin.com/in/pallavi1401
https://in.linkedin.com/in/parthasamai75
https://in.linkedin.com/in/preethamsingh


In [6]:
import re

urls = [
    "https://in.linkedin.com/in/harjeetkhanduja",
    "https://in.linkedin.com/in/mustafa-m-merchant",
    "https://in.linkedin.com/in/manishsinghhr",
    "https://in.linkedin.com/in/hr1505",
    "https://in.linkedin.com/in/rahulmu",
    "https://in.linkedin.com/in/sanjeev-varshney-0114b210",
    "https://in.linkedin.com/in/pallavi1401",
    "https://in.linkedin.com/in/parthasamai75",
    "https://in.linkedin.com/in/preethamsingh"
]

def extract_name(url):
    match = re.search(r'linkedin\.com/in/([a-z0-9\-]+)', url)
    if not match:
        return None

    slug = match.group(1)

    # Remove trailing alphanumeric IDs e.g. -0114b210
    slug = re.sub(r'-[0-9a-f]{6,}$', '', slug)
    # Remove trailing numbers e.g. 75, 1505
    slug = re.sub(r'\d+$', '', slug)

    parts = slug.replace("-", " ").strip().split()

    # Remove noise words
    noise = {"hr", "m", "mr", "ms", "dr", "in"}
    parts = [p for p in parts if p not in noise]

    # Slug with no dashes — try to split camelcase style
    # e.g. harjeetkhanduja → too merged, keep as is
    if len(parts) == 1 and len(parts[0]) > 4:
        # Can't reliably split — return as single capitalized
        return parts[0].capitalize()

    if len(parts) >= 2:
        return " ".join(p.capitalize() for p in parts[:2])

    return None  # Not enough info

for url in urls:
    name = extract_name(url)
    slug = url.split('/in/')[1]
    if name:
        print(f"  [✓] {name:<25} ← {slug}")
    else:
        print(f"  [✗] SKIP                    ← {slug} (unclear name)")

  [✓] Harjeetkhanduja           ← harjeetkhanduja
  [✓] Mustafa Merchant          ← mustafa-m-merchant
  [✓] Manishsinghhr             ← manishsinghhr
  [✗] SKIP                    ← hr1505 (unclear name)
  [✓] Rahulmu                   ← rahulmu
  [✓] Sanjeev Varshney          ← sanjeev-varshney-0114b210
  [✓] Pallavi                   ← pallavi1401
  [✓] Parthasamai               ← parthasamai75
  [✓] Preethamsingh             ← preethamsingh


In [7]:
import requests
from bs4 import BeautifulSoup

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
}

def fetch_name_from_page(url):
    try:
        resp = requests.get(url, headers=HEADERS, timeout=5)
        soup = BeautifulSoup(resp.text, "html.parser")
        title = soup.find("title")
        if title:
            # "Priya Sharma - HR Manager | LinkedIn"
            name = title.text.split("-")[0].strip()
            if len(name.split()) >= 2:
                return name
    except Exception as e:
        print(f"  [!] Error: {e}")
    return None

# Test on merged-name URLs only
test_urls = [
    "https://in.linkedin.com/in/harjeetkhanduja",
    "https://in.linkedin.com/in/manishsinghhr",
    "https://in.linkedin.com/in/rahulmu",
    "https://in.linkedin.com/in/pallavi1401",
]

for url in test_urls:
    name = fetch_name_from_page(url)
    print(f"  {name} ← {url.split('/in/')[1]}")

  Harjeet Khanduja ← harjeetkhanduja
  None ← manishsinghhr
  Rahul Mukherjee ← rahulmu
  None ← pallavi1401
